# 🎬 Anime Factory — Distributed Cloud Worker
Run cells **in order (1→9)**. GPU must be enabled: Runtime → Change runtime type → T4 GPU.

## Cell 1 — Mount Google Drive & Install Dependencies

In [ ]:
# CELL 1: Mount Drive & install dependencies
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import subprocess, sys
print('Installing system tools...')
subprocess.run(['apt-get', '-qq', 'install', '-y', 'aria2', 'ffmpeg'], check=True, capture_output=True)
print('Installing Python packages...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'requests'], check=True, capture_output=True)
print('✅ Cell 1 complete!')


## Cell 2 — Download & Install ComfyUI

In [ ]:
# CELL 2: Install ComfyUI (skip if already done)
import os
if not os.path.exists('/content/ComfyUI'):
    print('Cloning ComfyUI...')
    !git clone -q https://github.com/comfyanonymous/ComfyUI /content/ComfyUI
    %cd /content/ComfyUI
    !pip install -q -r requirements.txt
    print('✅ ComfyUI installed!')
else:
    print('✅ ComfyUI already present, skipping.')

# Create directories
os.makedirs('/content/ComfyUI/output', exist_ok=True)
os.makedirs('/content/ComfyUI/models/checkpoints', exist_ok=True)
os.makedirs('/content/ComfyUI/models/diffusion_models', exist_ok=True)
os.makedirs('/content/ComfyUI/models/text_encoders', exist_ok=True)
os.makedirs('/content/ComfyUI/models/vae', exist_ok=True)

# Clone VideoHelperSuite custom node for webm/video operations
custom_nodes_dir = '/content/ComfyUI/custom_nodes'
if not os.path.exists(os.path.join(custom_nodes_dir, 'ComfyUI-VideoHelperSuite')):
    print('Cloning ComfyUI-VideoHelperSuite...')
    !git clone -q https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite {custom_nodes_dir}/ComfyUI-VideoHelperSuite
    print('✅ VideoHelperSuite installed!')
else:
    print('✅ VideoHelperSuite already present.')


## Cell 3 — Download AI Models (~15 GB, takes ~7 min)

In [ ]:
# CELL 3: Download SDXL (AnythingXL) and Wan2.1 Models
import os

MODELS = {
    '/content/ComfyUI/models/checkpoints/AnythingXL_xl.safetensors':
        'https://civitai.com/api/download/models/384264?type=Model&format=SafeTensor',
    '/content/ComfyUI/models/vae/sdxl_vae.safetensors':
        'https://huggingface.co/stabilityai/sdxl-vae/resolve/main/sdxl_vae.safetensors',
    '/content/ComfyUI/models/diffusion_models/wan2.1_t2v_1.3B_fp16.safetensors':
        'https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/diffusion_models/wan2.1_t2v_1.3B_fp16.safetensors',
    '/content/ComfyUI/models/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors':
        'https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors',
    '/content/ComfyUI/models/vae/wan_2.1_vae.safetensors':
        'https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors'
}

for path, url in MODELS.items():
    if not os.path.exists(path):
        print(f'Downloading {os.path.basename(path)}...')
        !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
            "{url}" \
            -d "{os.path.dirname(path)}" \
            -o "{os.path.basename(path)}"
    else:
        print(f'✅ {os.path.basename(path)} already downloaded.')

# Verify
print('\nModel files status:')
for path in MODELS.keys():
    if os.path.exists(path):
        size = os.path.getsize(path) / 1e9
        print(f'  [FOUND] {os.path.basename(path)}  ({size:.2f} GB)')
    else:
        print(f'  [MISSING] {os.path.basename(path)}')


## Cell 4 — Start ComfyUI Server

In [ ]:
# CELL 4: Start ComfyUI server in background
import subprocess, time, urllib.request

proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '0.0.0.0', '--port', '8188',
     '--dont-print-server', '--disable-auto-launch'],
    cwd='/content/ComfyUI',
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

print('Waiting for ComfyUI to start...')
for i in range(24):   # up to 2 minutes
    time.sleep(5)
    try:
        urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=3)
        print('✅ ComfyUI is ready!')
        break
    except:
        print(f'  waiting... ({(i+1)*5}s)')
else:
    print('❌ ComfyUI failed to start! Check Cell 2 ran correctly.')


## Cell 5 — Write drive_state.py

In [ ]:
%%writefile /content/drive_state.py
"""
=============================================================
  DRIVE STATE MANAGER
  The distributed brain of the AI Video Factory.
=============================================================

This module handles ALL shared state through Google Drive.
It implements:
  - Scene claiming with file-based locks
  - Heartbeat system to detect crashed workers
  - Atomic writes (.tmp → .mp4) to prevent corruption
  - Progress tracking across multiple workers

Google Drive is used as both the database AND the file store.
Workers on Colab/Kaggle mount Drive directly and read/write files.
No network APIs needed — just filesystem operations.

Usage:
  state = DriveState("/content/drive/MyDrive/AnimeFactory")
  scene = state.claim_next_scene("worker_abc123")
  # ... generate video ...
  state.complete_scene(scene["id"], "/path/to/output.tmp")
"""

import json
import os
import shutil
import time
import threading
import uuid


class DriveState:
    """
    Manages distributed rendering state via Google Drive filesystem.
    
    Directory structure on Drive:
      AnimeFactory/
      ├── state/
      │   ├── master_script.json    ← Full story (read-only)
      │   ├── progress.json         ← Scene status tracking
      │   └── locks/                ← One .lock file per active scene
      ├── inputs/
      │   └── tts/                  ← Pre-generated audio files
      ├── outputs/
      │   └── scenes/              ← Completed scene videos
      └── logs/                    ← Per-worker log files
    """
    
    LOCK_STALE_SECONDS = 300  # 5 minutes = crashed worker
    HEARTBEAT_INTERVAL = 30   # Update lock every 30 seconds
    
    def __init__(self, drive_root):
        """
        Args:
            drive_root: Path to the AnimeFactory folder on mounted Drive.
                        e.g. "/content/drive/MyDrive/AnimeFactory"
        """
        self.root = drive_root
        self.state_dir = os.path.join(drive_root, "state")
        self.locks_dir = os.path.join(self.state_dir, "locks")
        self.inputs_dir = os.path.join(drive_root, "inputs")
        self.tts_dir = os.path.join(self.inputs_dir, "tts")
        self.outputs_dir = os.path.join(drive_root, "outputs")
        self.scenes_dir = os.path.join(self.outputs_dir, "scenes")
        self.logs_dir = os.path.join(drive_root, "logs")
        
        self.progress_file = os.path.join(self.state_dir, "progress.json")
        self.script_file = os.path.join(self.state_dir, "master_script.json")
        
        # Heartbeat thread reference (one per claimed scene)
        self._heartbeat_thread = None
        self._heartbeat_stop = threading.Event()
    
    def ensure_dirs(self):
        """Create the full directory structure on Drive."""
        for d in [self.state_dir, self.locks_dir, self.inputs_dir,
                  self.tts_dir, self.outputs_dir, self.scenes_dir, self.logs_dir]:
            os.makedirs(d, exist_ok=True)
    
    # ===========================================
    # PROGRESS FILE MANAGEMENT
    # ===========================================
    
    def load_progress(self):
        """Read progress.json from Drive. Returns dict."""
        if not os.path.exists(self.progress_file):
            return None
        with open(self.progress_file, "r") as f:
            return json.load(f)
    
    def save_progress(self, progress):
        """Write progress.json to Drive (atomic)."""
        progress["updated_at"] = time.time()
        tmp_path = self.progress_file + ".tmp"
        with open(tmp_path, "w") as f:
            json.dump(progress, f, indent=2)
        shutil.move(tmp_path, self.progress_file)
    
    def init_progress(self, scenes):
        """
        Initialize progress.json from a list of scenes.
        Called once by drive_uploader.py on your local PC.
        
        Args:
            scenes: List of scene dicts from master_script.json
        """
        progress = {
            "version": 1,
            "total_scenes": len(scenes),
            "scenes": {},
            "updated_at": time.time()
        }
        for i, scene in enumerate(scenes):
            scene_id = f"scene_{i+1:04d}"
            progress["scenes"][scene_id] = "pending"
        
        self.save_progress(progress)
        print(f"[STATE] Initialized {len(scenes)} scenes as 'pending'")
        return progress
    
    def load_script(self):
        """Read master_script.json from Drive."""
        with open(self.script_file, "r") as f:
            return json.load(f)
    
    # ===========================================
    # LOCK MANAGEMENT
    # ===========================================
    
    def _lock_path(self, scene_id):
        """Return the path to a scene's lock file."""
        return os.path.join(self.locks_dir, f"{scene_id}.lock")
    
    def _read_lock(self, scene_id):
        """Read a lock file. Returns dict or None if no lock exists."""
        path = self._lock_path(scene_id)
        if not os.path.exists(path):
            return None
        try:
            with open(path, "r") as f:
                return json.load(f)
        except (json.JSONDecodeError, IOError):
            return None
    
    def _write_lock(self, scene_id, worker_id):
        """Create or overwrite a lock file for a scene."""
        lock_data = {
            "worker_id": worker_id,
            "started_at": time.time(),
            "heartbeat": time.time()
        }
        path = self._lock_path(scene_id)
        with open(path, "w") as f:
            json.dump(lock_data, f)
        return lock_data
    
    def _update_heartbeat(self, scene_id):
        """Touch the lock file's heartbeat timestamp."""
        lock = self._read_lock(scene_id)
        if lock:
            lock["heartbeat"] = time.time()
            path = self._lock_path(scene_id)
            with open(path, "w") as f:
                json.dump(lock, f)
    
    def _is_lock_stale(self, scene_id):
        """
        Returns True if the lock is older than LOCK_STALE_SECONDS.
        A stale lock means the worker crashed and the scene should be reclaimed.
        """
        lock = self._read_lock(scene_id)
        if lock is None:
            return True  # No lock = available
        age = time.time() - lock.get("heartbeat", 0)
        return age > self.LOCK_STALE_SECONDS
    
    def _delete_lock(self, scene_id):
        """Remove a lock file."""
        path = self._lock_path(scene_id)
        if os.path.exists(path):
            os.remove(path)
    
    # ===========================================
    # HEARTBEAT THREAD
    # ===========================================
    
    def _start_heartbeat(self, scene_id):
        """Start a background thread that updates the lock every 30 seconds."""
        self._heartbeat_stop.clear()
        
        def _beat():
            while not self._heartbeat_stop.is_set():
                self._update_heartbeat(scene_id)
                self._heartbeat_stop.wait(self.HEARTBEAT_INTERVAL)
        
        self._heartbeat_thread = threading.Thread(target=_beat, daemon=True)
        self._heartbeat_thread.start()
    
    def _stop_heartbeat(self):
        """Stop the heartbeat thread."""
        self._heartbeat_stop.set()
        if self._heartbeat_thread:
            self._heartbeat_thread.join(timeout=5)
            self._heartbeat_thread = None
    
    # ===========================================
    # SCENE CLAIMING + RELEASING
    # ===========================================
    
    def claim_next_scene(self, worker_id):
        """
        Find the next unclaimed scene and lock it for this worker.
        
        Returns:
            dict with scene info: {"id": "scene_0005", "index": 4, "data": {...}}
            or None if no scenes are available.
        """
        progress = self.load_progress()
        if not progress:
            print("[STATE] No progress.json found!")
            return None
        
        script = self.load_script()
        
        for scene_id, status in progress["scenes"].items():
            if status == "done":
                continue
            
            # Check if scene is locked by another worker
            if status == "locked":
                if not self._is_lock_stale(scene_id):
                    continue  # Another worker is actively processing this
                else:
                    # Stale lock — crashed worker. Steal it.
                    old_lock = self._read_lock(scene_id)
                    old_worker = old_lock.get("worker_id", "unknown") if old_lock else "unknown"
                    print(f"[STATE] Stealing stale lock on {scene_id} from crashed worker {old_worker}")
            
            # Claim this scene
            self._write_lock(scene_id, worker_id)
            progress["scenes"][scene_id] = "locked"
            self.save_progress(progress)
            
            # Start heartbeat
            self._start_heartbeat(scene_id)
            
            # Get scene data from script
            idx = int(scene_id.split("_")[1]) - 1
            scene_data = script[idx] if idx < len(script) else {}
            
            print(f"[STATE] Worker {worker_id} claimed {scene_id}")
            return {
                "id": scene_id,
                "index": idx,
                "data": scene_data
            }
        
        print("[STATE] No more scenes to process!")
        return None
    
    def complete_scene(self, scene_id, tmp_output_path):
        """
        Mark a scene as done after successful generation.
        
        1. Rename .tmp → .mp4 (atomic)
        2. Update progress.json
        3. Delete lock file
        4. Stop heartbeat
        """
        self._stop_heartbeat()
        
        # Atomic rename: .tmp → .mp4
        final_path = os.path.join(self.scenes_dir, f"{scene_id}.mp4")
        if os.path.exists(tmp_output_path):
            shutil.move(tmp_output_path, final_path)
            print(f"[STATE] Atomic write: {scene_id}.mp4 ({os.path.getsize(final_path) / 1024 / 1024:.1f} MB)")
        
        # Update progress
        progress = self.load_progress()
        progress["scenes"][scene_id] = "done"
        self.save_progress(progress)
        
        # Clean up lock
        self._delete_lock(scene_id)
        
        # Count progress
        done = sum(1 for s in progress["scenes"].values() if s == "done")
        total = progress["total_scenes"]
        print(f"[STATE] {scene_id} complete! Progress: {done}/{total} ({done/total*100:.1f}%)")
        
        return final_path
    
    def fail_scene(self, scene_id):
        """
        Release a scene back to 'pending' after a failure.
        Another worker can pick it up later.
        """
        self._stop_heartbeat()
        
        # Reset to pending
        progress = self.load_progress()
        progress["scenes"][scene_id] = "pending"
        self.save_progress(progress)
        
        # Clean up lock and any .tmp files
        self._delete_lock(scene_id)
        tmp_path = os.path.join(self.scenes_dir, f"{scene_id}.tmp")
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
        
        print(f"[STATE] {scene_id} released back to 'pending'")
    
    def cleanup_stale(self):
        """
        Clean up any leftover .tmp files and stale locks from crashed workers.
        Called at worker startup.
        """
        cleaned = 0
        
        # Clean .tmp files
        if os.path.exists(self.scenes_dir):
            for f in os.listdir(self.scenes_dir):
                if f.endswith(".tmp"):
                    os.remove(os.path.join(self.scenes_dir, f))
                    scene_id = f.replace(".tmp", "")
                    cleaned += 1
        
        # Clean stale locks and reset their scenes to pending
        progress = self.load_progress()
        if progress and os.path.exists(self.locks_dir):
            for f in os.listdir(self.locks_dir):
                if f.endswith(".lock"):
                    scene_id = f.replace(".lock", "")
                    if self._is_lock_stale(scene_id):
                        self._delete_lock(scene_id)
                        if scene_id in progress["scenes"]:
                            progress["scenes"][scene_id] = "pending"
                        cleaned += 1
            
            if cleaned > 0:
                self.save_progress(progress)
        
        if cleaned > 0:
            print(f"[STATE] Cleaned up {cleaned} stale items from crashed workers")
        else:
            print("[STATE] No stale items found. Clean start.")
    
    def get_summary(self):
        """Print a summary of the current pipeline status."""
        progress = self.load_progress()
        if not progress:
            print("[STATE] No progress file found.")
            return
        
        total = progress["total_scenes"]
        counts = {"done": 0, "pending": 0, "locked": 0}
        for status in progress["scenes"].values():
            counts[status] = counts.get(status, 0) + 1
        
        print(f"\n{'='*50}")
        print(f"  Pipeline Status")
        print(f"{'='*50}")
        print(f"  Total scenes:   {total}")
        print(f"  Done:           {counts['done']}")
        print(f"  Pending:        {counts['pending']}")
        print(f"  In Progress:    {counts['locked']}")
        print(f"  Progress:       {counts['done']/total*100:.1f}%")
        print(f"{'='*50}\n")

## Cell 6 — Write drive_worker.py

In [ ]:
%%writefile /content/drive_worker.py
"""
=============================================================
  DRIVE WORKER — Stateless Rendering Engine
  Runs on Google Colab or Kaggle. Processes one scene at a time.
=============================================================

This script is the workhorse of the distributed pipeline.
It runs in a loop:
  1. Claims the next unclaimed scene from Google Drive
  2. Generates visuals via ComfyUI (running locally on the GPU)
  3. Merges audio + video with FFmpeg
  4. Writes output atomically (.tmp → .mp4) to Drive
  5. Repeats until no scenes remain

The worker is FULLY STATELESS:
  - It can be killed at any time without data loss
  - Multiple workers can run simultaneously (different Colab/Kaggle sessions)
  - A new worker automatically picks up where a crashed one left off

Usage (called from a Colab/Kaggle notebook cell):
  !python drive_worker.py --drive-path /content/drive/MyDrive/AnimeFactory
"""

import argparse
import json
import os
import random
import subprocess
import sys
import time
import urllib.request
import uuid

# Add the project to Python path
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from drive_state import DriveState


# ===========================================================
# CONFIGURATION
# ===========================================================

COMFYUI_URL = "http://127.0.0.1:8188"  # Local ComfyUI on the same GPU machine

# Manhwa style for all generations
STYLE_PREFIX = "manhwa style, webtoon art, sharp linework, vibrant colors, ultra detailed, masterpiece, best quality, "
NEGATIVE_PROMPT = "low quality, worst quality, blurry, bad anatomy, bad proportions, deformed, ugly, 3d render, photorealistic, watermark, text, signature, extra limbs"

# Voice map not needed here — TTS is pre-generated by drive_uploader.py


# ===========================================================
# COMFYUI API HELPERS
# ===========================================================

def api_post(url, data):
    """Send a JSON POST request to ComfyUI."""
    body = json.dumps(data).encode("utf-8")
    req = urllib.request.Request(url, data=body, headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=300) as resp:
            return json.loads(resp.read().decode("utf-8"))
    except Exception as e:
        print(f"  [ERROR] API call to {url} failed: {e}")
        return None


def api_get(url):
    """Send a GET request to ComfyUI."""
    try:
        with urllib.request.urlopen(url, timeout=30) as resp:
            return json.loads(resp.read().decode("utf-8"))
    except Exception as e:
        return None


def wait_for_comfyui_ready():
    """Wait until ComfyUI is responding on its API port."""
    print("[WORKER] Waiting for ComfyUI to be ready...")
    for i in range(60):  # Wait up to 5 minutes
        result = api_get(f"{COMFYUI_URL}/system_stats")
        if result:
            print("[WORKER] ComfyUI is ready!")
            return True
        time.sleep(5)
    print("[WORKER] ERROR: ComfyUI did not start in time!")
    return False


# ===========================================================
# IMAGE GENERATION (AnythingXL)
# ===========================================================

def generate_image(prompt, scene_id, comfyui_output_dir):
    """Generate a manhwa-style image via ComfyUI AnythingXL."""
    full_prompt = STYLE_PREFIX + prompt
    unique_prefix = scene_id
    
    workflow = {
        "1": {
            "inputs": {"ckpt_name": "AnythingXL_xl.safetensors"},
            "class_type": "CheckpointLoaderSimple"
        },
        "2": {
            "inputs": {"text": full_prompt, "clip": ["1", 1]},
            "class_type": "CLIPTextEncode"
        },
        "3": {
            "inputs": {"text": NEGATIVE_PROMPT, "clip": ["1", 1]},
            "class_type": "CLIPTextEncode"
        },
        "4": {
            "inputs": {"width": 1216, "height": 832, "batch_size": 1},
            "class_type": "EmptyLatentImage"
        },
        "5": {
            "inputs": {
                "seed": random.randint(1, 2**32),
                "steps": 25, "cfg": 7.0,
                "sampler_name": "euler", "scheduler": "karras",
                "denoise": 1.0,
                "model": ["1", 0], "positive": ["2", 0],
                "negative": ["3", 0], "latent_image": ["4", 0]
            },
            "class_type": "KSampler"
        },
        "6": {
            "inputs": {"samples": ["5", 0], "vae": ["1", 2]},
            "class_type": "VAEDecode"
        },
        "7": {
            "inputs": {"filename_prefix": unique_prefix, "images": ["6", 0]},
            "class_type": "SaveImage"
        }
    }
    
    print(f"  [IMG] Generating image for {scene_id}...")
    result = api_post(f"{COMFYUI_URL}/prompt", {"prompt": workflow})
    
    if not result or "prompt_id" not in result:
        print(f"  [IMG] FAILED to queue {scene_id}")
        return None
    
    return wait_for_output(result["prompt_id"], unique_prefix, comfyui_output_dir)


# ===========================================================
# VIDEO GENERATION (Wan2.1)
# ===========================================================

def generate_video(prompt, scene_id, comfyui_output_dir, part=1, fps=16):
    """Generate a video clip via ComfyUI Wan2.1."""
    full_prompt = STYLE_PREFIX + prompt
    unique_prefix = f"{scene_id}_p{part}"
    
    wan_negative = "low quality, worst quality, blurry, static, distorted, 3d render, photorealistic, ugly"
    
    workflow = {
        "37": {
            "inputs": {
                "unet_name": "wan2.1_t2v_1.3B_fp16.safetensors",
                "weight_dtype": "default"
            },
            "class_type": "UNETLoader"
        },
        "38": {
            "inputs": {
                "clip_name": "umt5_xxl_fp8_e4m3fn_scaled.safetensors",
                "type": "wan",
                "device": "default"
            },
            "class_type": "CLIPLoader"
        },
        "39": {
            "inputs": {"vae_name": "wan_2.1_vae.safetensors"},
            "class_type": "VAELoader"
        },
        "48": {
            "inputs": {"model": ["37", 0], "shift": 8},
            "class_type": "ModelSamplingSD3"
        },
        "6": {
            "inputs": {"text": full_prompt, "clip": ["38", 0]},
            "class_type": "CLIPTextEncode"
        },
        "7": {
            "inputs": {"text": wan_negative, "clip": ["38", 0]},
            "class_type": "CLIPTextEncode"
        },
        "40": {
            "inputs": {"width": 832, "height": 480, "length": 81, "batch_size": 1},
            "class_type": "EmptyHunyuanLatentVideo"
        },
        "3": {
            "inputs": {
                "seed": random.randint(1, 2**32),
                "steps": 30, "cfg": 6.0,
                "sampler_name": "uni_pc", "scheduler": "simple",
                "denoise": 1.0,
                "model": ["48", 0], "positive": ["6", 0],
                "negative": ["7", 0], "latent_image": ["40", 0]
            },
            "class_type": "KSampler"
        },
        "8": {
            "inputs": {"samples": ["3", 0], "vae": ["39", 0]},
            "class_type": "VAEDecode"
        },
        "28": {
            "inputs": {
                "filename_prefix": unique_prefix,
                "codec": "vp9", "fps": fps, "crf": 32,
                "images": ["8", 0]
            },
            "class_type": "SaveWEBM"
        }
    }
    
    print(f"  [VIDEO] Generating {scene_id} part {part} @ {fps} FPS...")
    result = api_post(f"{COMFYUI_URL}/prompt", {"prompt": workflow})
    
    if not result or "prompt_id" not in result:
        print(f"  [VIDEO] FAILED to queue {scene_id} part {part}")
        return None
    
    return wait_for_output(result["prompt_id"], unique_prefix, comfyui_output_dir)


def wait_for_output(prompt_id, prefix, comfyui_output_dir):
    """Poll ComfyUI until generation is done, then find the output file."""
    max_wait = 900  # 15 minutes
    elapsed = 0
    
    while elapsed < max_wait:
        time.sleep(5)
        elapsed += 5
        
        history = api_get(f"{COMFYUI_URL}/history/{prompt_id}")
        if history and prompt_id in history:
            print(f"  [DONE] ComfyUI finished in {elapsed}s")
            break
    else:
        print(f"  [TIMEOUT] ComfyUI took too long for {prefix}")
        return None
    
    # Find output file
    matching = []
    for f in os.listdir(comfyui_output_dir):
        if f.startswith(prefix) and not f.endswith(".webp"):
            matching.append(os.path.join(comfyui_output_dir, f))
    
    if matching:
        matching.sort(key=os.path.getmtime, reverse=True)
        print(f"  [FOUND] {matching[0]}")
        return matching[0]
    
    print(f"  [ERROR] No output file found for {prefix}")
    return None


# ===========================================================
# SCENE PROCESSING (One scene at a time)
# ===========================================================

def process_scene(scene_info, state, comfyui_output_dir):
    """
    Process a single scene: generate visuals, merge with audio, save.
    
    Args:
        scene_info: Dict from DriveState.claim_next_scene()
        state: DriveState instance
        comfyui_output_dir: ComfyUI's output directory on this machine
    
    Returns:
        Path to the final .tmp output, or None on failure.
    """

    scene_id = scene_info["id"]
    scene_data = scene_info["data"]
    scene_type = scene_data.get("type", "image")
    
    # Resolve the correct prompt based on scene type for character and style consistency
    if scene_type == "video":
        visual_prompt = (
            scene_data.get("video_prompt") 
            or scene_data.get("visual_prompt") 
            or scene_data.get("image_prompt") 
            or scene_data.get("text") 
            or scene_data.get("story_text") 
            or ""
        )
    else:
        visual_prompt = (
            scene_data.get("image_prompt") 
            or scene_data.get("visual_prompt") 
            or scene_data.get("text") 
            or scene_data.get("story_text") 
            or ""
        )
    
    # Get pre-generated audio from Drive
    audio_path = os.path.join(state.tts_dir, f"{scene_id}.mp3")
    if not os.path.exists(audio_path):
        print(f"  [ERROR] Missing TTS audio: {audio_path}")
        return None
    
    # Generate visual(s)
    if scene_type == "video":
        visual_path = generate_video_scene(visual_prompt, scene_id, comfyui_output_dir)
    else:
        visual_path = generate_image(visual_prompt, scene_id, comfyui_output_dir)
    
    if not visual_path:
        return None
    
    # Merge audio + visual into scene video
    tmp_output = os.path.join(state.scenes_dir, f"{scene_id}.tmp")
    if merge_scene(visual_path, audio_path, tmp_output, scene_type == "image"):
        return tmp_output
    
    return None


def generate_video_scene(prompt, scene_id, comfyui_output_dir):
    """
    Generate dual-clip video scene (wide shot + close up).
    Concatenates both parts with FFmpeg.
    """
    # Dynamic FPS pacing
    rand_val = random.random()
    if rand_val < 0.50:
        fps = 24
    elif rand_val < 0.85:
        fps = 16
    else:
        fps = 8
    
    print(f"  [VIDEO] Dynamic pacing: {fps} FPS for {scene_id}")
    
    # Part 1: Wide shot
    path_1 = generate_video(prompt + ", wide shot, full body, environment visible",
                            scene_id, comfyui_output_dir, part=1, fps=fps)
    if not path_1:
        return None
    
    # Part 2: Close up
    path_2 = generate_video(prompt + ", dramatic close up shot, detailed face",
                            scene_id, comfyui_output_dir, part=2, fps=fps)
    if not path_2:
        return path_1  # Use just part 1 if part 2 fails
    
    # Concatenate both parts
    concat_path = os.path.join(comfyui_output_dir, f"{scene_id}_concat.webm")
    concat_txt = os.path.join(comfyui_output_dir, f"{scene_id}_concat.txt")
    
    with open(concat_txt, "w") as f:
        f.write(f"file '{os.path.abspath(path_1).replace(chr(92), '/')}'\n")
        f.write(f"file '{os.path.abspath(path_2).replace(chr(92), '/')}'\n")
    
    try:
        subprocess.run(
            ["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", concat_txt,
             "-c", "copy", concat_path],
            stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT, check=True
        )
        print(f"  [VIDEO] Concatenated {scene_id}: 2 clips merged")
        return concat_path
    except subprocess.CalledProcessError:
        print(f"  [ERROR] FFmpeg concat failed for {scene_id}")
        return path_1  # Fallback to just part 1


def merge_scene(visual_path, audio_path, output_path, is_image):
    """
    Merge visual + audio into a final scene video using FFmpeg.
    Uses simple scale+pad for maximum compatibility.
    """
    if is_image:
        # -loop 1 makes FFmpeg loop the single image for the duration of the audio
        command = [
            "ffmpeg", "-y",
            "-loop", "1", "-i", visual_path,
            "-i", audio_path,
            "-filter_complex",
                "[0:v]scale=1920:1080:force_original_aspect_ratio=decrease,"
                "pad=1920:1080:(ow-iw)/2:(oh-ih)/2:color=black,"
                "format=yuv420p[final]",
            "-map", "[final]", "-map", "1:a",
            "-c:v", "libx264", "-preset", "fast", "-tune", "stillimage",
            "-c:a", "aac", "-b:a", "192k",
            "-shortest", output_path
        ]
    else:
        command = [
            "ffmpeg", "-y",
            "-i", visual_path,
            "-i", audio_path,
            "-filter_complex",
                "[0:v]scale=1920:1080:force_original_aspect_ratio=decrease,"
                "pad=1920:1080:(ow-iw)/2:(oh-ih)/2:color=black,"
                "format=yuv420p[final]",
            "-map", "[final]", "-map", "1:a",
            "-c:v", "libx264", "-preset", "fast",
            "-c:a", "aac", "-b:a", "192k",
            "-shortest", output_path
        ]
    
    try:
        result = subprocess.run(command, check=True, capture_output=True)
        size = os.path.getsize(output_path) / 1024 / 1024
        print(f"  [MERGE] Scene merged: {size:.1f} MB")
        return True
    except subprocess.CalledProcessError as e:
        print(f"  [MERGE] FAILED: {e.stderr.decode()[:400]}")
        return False


# ===========================================================
# MAIN WORKER LOOP
# ===========================================================

def run_worker(drive_path, comfyui_output_dir, max_scenes=None):
    """
    Main worker loop. Runs until no more scenes are available.
    
    Args:
        drive_path: Path to AnimeFactory on mounted Google Drive
        comfyui_output_dir: ComfyUI's local output directory
        max_scenes: Optional limit on scenes to process this session
    """
    worker_id = f"worker_{uuid.uuid4().hex[:8]}"
    state = DriveState(drive_path)
    
    print(f"\n{'='*60}")
    print(f"  DISTRIBUTED WORKER: {worker_id}")
    print(f"  Drive: {drive_path}")
    print(f"  ComfyUI: {COMFYUI_URL}")
    print(f"{'='*60}\n")
    
    # Wait for ComfyUI
    if not wait_for_comfyui_ready():
        return
    
    # Clean up any leftovers from crashed workers
    state.cleanup_stale()
    state.get_summary()
    
    scenes_processed = 0
    
    while True:
        # Check scene limit
        if max_scenes and scenes_processed >= max_scenes:
            print(f"\n[WORKER] Reached limit of {max_scenes} scenes. Stopping.")
            break
        
        # Claim next scene
        scene_info = state.claim_next_scene(worker_id)
        if not scene_info:
            print("\n[WORKER] No more scenes to process. Worker complete!")
            break
        
        scene_id = scene_info["id"]
        print(f"\n{'='*50}")
        print(f"  Processing: {scene_id}")
        print(f"  Type: {scene_info['data'].get('type', 'image')}")
        print(f"  Text: {scene_info['data'].get('text', '')[:80]}...")
        print(f"{'='*50}")
        
        try:
            # Process the scene
            tmp_output = process_scene(scene_info, state, comfyui_output_dir)
            
            if tmp_output and os.path.exists(tmp_output):
                # Success! Atomic write to Drive
                state.complete_scene(scene_id, tmp_output)
                scenes_processed += 1
            else:
                # Generation failed — release scene for retry
                state.fail_scene(scene_id)
        
        except KeyboardInterrupt:
            print(f"\n[WORKER] Interrupted! Releasing {scene_id}...")
            state.fail_scene(scene_id)
            break
        
        except Exception as e:
            print(f"\n[WORKER] Unexpected error on {scene_id}: {e}")
            state.fail_scene(scene_id)
            # Continue to next scene instead of crashing
            continue
    
    # Final summary
    state.get_summary()
    print(f"\n[WORKER] {worker_id} processed {scenes_processed} scenes. Goodbye!")


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Distributed Scene Worker")
    parser.add_argument("--drive-path", type=str, required=True,
                        help="Path to AnimeFactory on mounted Drive")
    parser.add_argument("--comfyui-output", type=str, 
                        default="/content/ComfyUI/output",
                        help="ComfyUI's local output directory")
    parser.add_argument("--max-scenes", type=int, default=None,
                        help="Max scenes to process (optional)")
    
    args = parser.parse_args()
    run_worker(args.drive_path, args.comfyui_output, args.max_scenes)

## Cell 7 — Check Drive & Initialize Progress

In [ ]:
# CELL 7: Verify Drive structure and create progress.json if needed
import json, os, time

DRIVE_ROOT = '/content/drive/MyDrive/AnimeFactory'

# Create all required folders
folders = [
    'state/locks', 'inputs/tts', 'outputs/scenes', 'logs'
]
for f in folders:
    os.makedirs(os.path.join(DRIVE_ROOT, f), exist_ok=True)
print('✅ Folder structure ready')

# Check master_script.json exists
script_path = os.path.join(DRIVE_ROOT, 'state/master_script.json')
if not os.path.exists(script_path):
    print('❌ ERROR: master_script.json not found!')
    print(f'   Upload it to: {script_path}')
else:
    with open(script_path) as f:
        scenes = json.load(f)
    print(f'✅ Script loaded: {len(scenes)} scenes')

    # Check TTS files
    tts_dir = os.path.join(DRIVE_ROOT, 'inputs/tts')
    tts_files = [f for f in os.listdir(tts_dir) if f.endswith('.mp3')]
    print(f'✅ TTS audio files: {len(tts_files)}/{len(scenes)} found')
    if len(tts_files) < len(scenes):
        print('⚠️  Missing audio! Run drive_uploader.py on your PC first.')

    # Initialize progress.json if missing
    progress_path = os.path.join(DRIVE_ROOT, 'state/progress.json')
    if not os.path.exists(progress_path):
        progress = {
            'version': 1,
            'total_scenes': len(scenes),
            'scenes': {f'scene_{i+1:04d}': 'pending' for i in range(len(scenes))},
            'updated_at': time.time()
        }
        with open(progress_path, 'w') as f:
            json.dump(progress, f, indent=2)
        print(f'✅ Initialized progress.json ({len(scenes)} scenes as pending)')
    else:
        with open(progress_path) as f:
            p = json.load(f)
        done = sum(1 for s in p['scenes'].values() if s == 'done')
        print(f'✅ progress.json exists: {done}/{len(scenes)} scenes done')


## Cell 8 — 🚀 Run the Worker (Main Loop)

In [ ]:
# CELL 8: Start the distributed worker loop
# This will run until all scenes are done or the session expires.
import sys
sys.path.insert(0, '/content')

DRIVE_ROOT = '/content/drive/MyDrive/AnimeFactory'
COMFYUI_OUTPUT = '/content/ComfyUI/output'

from drive_worker import run_worker
run_worker(DRIVE_ROOT, COMFYUI_OUTPUT)


## Cell 9 — 🎞️ Final Stitch (Run ONLY when all scenes are done)

In [ ]:
# CELL 9: Stitch all completed scenes into final video
import os, json, subprocess

DRIVE_ROOT = '/content/drive/MyDrive/AnimeFactory'
scenes_dir = os.path.join(DRIVE_ROOT, 'outputs', 'scenes')
final_path = os.path.join(DRIVE_ROOT, 'outputs', 'final_video.mp4')

scene_files = sorted([
    os.path.join(scenes_dir, f)
    for f in os.listdir(scenes_dir)
    if f.endswith('.mp4')
])

print(f'Found {len(scene_files)} completed scenes')

list_file = os.path.join(DRIVE_ROOT, 'outputs', 'concat_list.txt')
with open(list_file, 'w') as f:
    for sf in scene_files:
        f.write(f"file '{sf}'\n")

print('Stitching final video...')
subprocess.run([
    'ffmpeg', '-y', '-f', 'concat', '-safe', '0',
    '-i', list_file, '-c', 'copy', final_path
], check=True)

size = os.path.getsize(final_path) / 1e6
print(f'✅ Final video saved: {final_path} ({size:.1f} MB)')
